In [1]:
import torch_openreml
print(torch_openreml.example_data.john_alpha)

    plot rep block  gen   yield  row  col
0      1  R1    B1  G11  4.1172    1    1
1      2  R1    B1  G04  4.4461    2    1
2      3  R1    B1  G05  5.8757    3    1
3      4  R1    B1  G22  4.5784    4    1
4      5  R1    B2  G21  4.6540    5    1
..   ...  ..   ...  ...     ...  ...  ...
67    68  R3    B5  G24  3.5655   68    1
68    69  R3    B6  G03  2.8873   69    1
69    70  R3    B6  G05  4.1972   70    1
70    71  R3    B6  G20  3.7349   71    1
71    72  R3    B6  G07  3.6096   72    1

[72 rows x 7 columns]


In [2]:
import torch
import pandas as pd
from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, IdentityMatrix, ScalarMatrix, KroneckerProduct, Augment
from torch_openreml.example_data import john_alpha

In [3]:
rep = john_alpha["rep"]
block = john_alpha["block"]
gen = john_alpha["gen"]
n = len(john_alpha)

y = torch.tensor(john_alpha["yield"].values)
X = Augment(torch.ones(n, 1), DummyMatrix(rep, drop_first=True))()

In [4]:
reml = MixedModelREML(
    genotype = (DummyMatrix(gen), ScalarMatrix(gen)),
    replicate_block = (DummyMatrix(rep, block), KroneckerProduct(
        G_rep = IdentityMatrix(rep),
        G_block = ScalarMatrix(block),
    )),
    R = ScalarMatrix(n)
)

In [5]:
print(reml)

MarginalCovariance(
  operands={
    'Z': Augment(
      operands={
        'genotype': DummyMatrix(shape=(72, 24)),
        'replicate_block': DummyMatrix(shape=(72, 18))
      }
    ),
    'G': BlockDiagonal(
      operands={
        'genotype': ScalarMatrix(shape=(24, 24), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}}),
        'replicate_block': KroneckerProduct(
          operands={
            'G_rep': IdentityMatrix(shape=(3, 3)),
            'G_block': ScalarMatrix(shape=(6, 6), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}})
          }
        )
      }
    ),
    'R': ScalarMatrix(shape=(72, 72), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}})
  }
)


In [6]:
free_param_names, free_params = reml.optimize(y, X, torch.zeros(3), verbose=1)
print(free_param_names)
print(free_params)
print(reml.build_params(free_params))

[  1] ∥∇∥:      41.8198, ∥Δ∥: 8.9438, η: 0.25, ∥Δᶜ∥: 2.2359, accepted: True , log 𝓛:  13.4244
[  2] ∥∇∥:      19.0068, ∥Δ∥: 0.4705, η: 1.00, ∥Δᶜ∥: 0.4705, accepted: True , log 𝓛:  16.6687
[  3] ∥∇∥:       3.6187, ∥Δ∥: 0.0932, η: 1.00, ∥Δᶜ∥: 0.0932, accepted: True , log 𝓛:  16.7988
[  4] ∥∇∥:       0.1519, ∥Δ∥: 0.0042, η: 1.00, ∥Δᶜ∥: 0.0042, accepted: True , log 𝓛:  16.8141
[  5] ∥∇∥:       0.0020, ∥Δ∥: 0.0001, η: 0.00, ∥Δᶜ∥: 0.0000, accepted: False, log 𝓛:  16.8081
[  6] ∥∇∥:       0.0019, ∥Δ∥: 0.0001, η: 1.00, ∥Δᶜ∥: 0.0001, accepted: True , log 𝓛:  16.8137
[  7] ∥∇∥:       0.0001, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  16.8157
[  8] ∥∇∥:       0.0001, ∥Δ∥: 0.0000, η: 0.02, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  16.8157

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 8, total time: 0.03s

['G/genotype/sigma^2', 'G/replicate_block/G_block/sigma^2', 'R/sigma^2']
tensor([-0.9728, -1.3281, -1.2529]

In [7]:
beta_hat = reml.blue()
print(beta_hat)

tensor([ 4.5183,  0.2978, -0.4140])


In [8]:
b_hat = reml.blup()
b_hat

tensor([ 0.5012,  0.0050, -0.7846,  0.0061,  0.4749,  0.0446, -0.3089,  0.0622,
        -0.8099, -0.0894, -0.1964,  0.2258,  0.2317,  0.2434,  0.4247,  0.2010,
         0.0781, -0.1102,  0.2896, -0.3390,  0.2561,  0.0241, -0.1770, -0.2531,
         0.1231, -0.1412, -0.1504, -0.1068,  0.0737,  0.2015, -0.5326, -0.3012,
         0.2432,  0.1349,  0.2753,  0.1804,  0.0506, -0.0478,  0.1511,  0.0538,
        -0.0080, -0.1996])

In [9]:
gen_blup = reml.blup_per_term()
print(gen_blup["genotype"])

tensor([ 0.5012,  0.0050, -0.7846,  0.0061,  0.4749,  0.0446, -0.3089,  0.0622,
        -0.8099, -0.0894, -0.1964,  0.2258,  0.2317,  0.2434,  0.4247,  0.2010,
         0.0781, -0.1102,  0.2896, -0.3390,  0.2561,  0.0241, -0.1770, -0.2531])
